In [56]:
import pickle
import glob
import os
import subprocess
import pandas as pd
import numpy as np


In [ ]:

odb_filenames = glob.glob('abq\\*.odb')
print('found odbs:', odb_filenames)


for odb_filename in odb_filenames:
    # abaqus python extract_from_odb.py .\abq\Job-2.odb --processes 32 --field_names NT11 S
    result = subprocess.run(["abaqus", "python", "extract_from_odb.py", odb_filename, "--processes", "16", "--field_names", "NT11", "S"], shell=True, capture_output=True, text=True)
    
    print(result)

In [2]:
data_files = glob.glob('abq\\Job-1\\*.pkl')

print(data_files)

data =[]

for data_file in data_files:
    df = pd.read_pickle(data_file)
    data.append(df)

print(len(data))

['abq\\Job-1\\Job-1_step_1_inc_0_field_COORD_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_COORD_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_COORD_block_2.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_COORD_block_3.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_NT11_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_NT11_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_S_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_0_field_S_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_COORD_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_COORD_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_COORD_block_2.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_COORD_block_3.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_NT11_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_NT11_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_S_block_0.pkl', 'abq\\Job-1\\Job-1_step_1_inc_10_field_S_block_1.pkl', 'abq\\Job-1\\Job-1_step_1_inc_1_field_COORD_block_0.pkl', 'abq\\Job-1\\Job-1_step_1

In [14]:
data[-1]

field_name                                                S           \
component                                               S11            
instance_name                                      PART-1-2            
idx                                                  (1, 1)   (2, 1)   
step_number increment_number total_time step_name                      
2           4                12.0       final      0.001863  0.00318   

field_name                                                             \
component                                                               
instance_name                                                           
idx                                                  (3, 1)    (4, 1)   
step_number increment_number total_time step_name                       
2           4                12.0       final      0.003013 -0.007578   

field_name                                                             \
component                                                               
instance_name                                                           
idx                                                  (5, 1)    (6, 1)   
step_number increment_number total_time step_name                       
2           4                12.0       final     -0.007044 -0.009159   

field_name                                                            \
component                                                              
instance_name                                                          
idx                                                  (7, 1)   (8, 1)   
step_number increment_number total_time step_name                      
2           4                12.0       final     -0.007682  0.00229   

field_name                                                             ...  \
component                                                              ...   
instance_name                                                          ...   
idx                                                  (9, 1)   (10, 1)  ...   
step_number increment_number total_time step_name                      ...   
2           4                12.0       final      0.009754  0.001512  ...   

field_name                                                             \
component                                               S23             
instance_name                                      PART-1-2             
idx                                                (119, 1)  (120, 1)   
step_number increment_number total_time step_name                       
2           4                12.0       final      0.023812 -0.040767   

field_name                                                             \
component                                                               
instance_name                                                           
idx                                                (121, 1)  (122, 1)   
step_number increment_number total_time step_name                       
2           4                12.0       final      0.027191 -0.104398   

field_name                                                             \
component                                                               
instance_name                                                           
idx                                                (123, 1)  (124, 1)   
step_number increment_number total_time step_name                       
2           4                12.0       final     -0.099942 -0.058206   

field_name                                                            \
component                                                              
instance_name                                                          
idx                                               (125, 1)  (126, 1)   
step_number increment_number total_time step_name                      
2           4                12.0       final      0.11523  0.118797   

field_name                           

In [8]:
df.index

MultiIndex([(1, 0, 0.0, 'trans')],
           names=['step_number', 'increment_number', 'total_time', 'step_name'])

In [57]:
inds = [df.index[0]    for df in data]
field_name = [df.columns.get_level_values(0)[0] for df in data]

df_meta = pd.DataFrame(inds, columns = df.index.names)
df_meta['field_name'] = field_name
G = df_meta.groupby(by=df_meta.columns.tolist())

data_dict = {}
for name, group in G:

    indicies = group.index.to_list()

    df_tmp = pd.concat([data[i] for i in indicies], axis = 1)
    field_name = df_tmp.columns.get_level_values(0)[0]

    if field_name not in data_dict:
        data_dict[field_name] = []
    data_dict[field_name].append(df_tmp)

for field_name, df_list in data_dict.items():

    df_tmp = pd.concat(df_list, axis = 0)
    df_tmp.sort_index(inplace = True, axis = 0)
    df_tmp.sort_index(inplace = True, axis = 1)
    df_tmp = df_tmp.astype(np.float32)
    assert df_tmp.notna().all().all(), "Some values are missing"


    data_dict[field_name] = df_tmp


C:\Users\ssmee\AppData\Local\Temp\ipykernel_249196\1280837643.py:13: RuntimeWarning: '<' not supported between instances of 'int' and 'tuple', sort order is undefined for incomparable objects.
  df_tmp = pd.concat([data[i] for i in indicies], axis = 1)


In [61]:
for field_name, df in data_dict.items():
    print(df.head())

    df.to_csv(f'{field_name}.csv')

field_name                                            COORD            \
component                                             COOR1             
instance_name                                      PART-1-1             
idx                                                       1         2   
step_number increment_number total_time step_name                       
1           0                0.0        trans      9.009688  6.234898   
            1                1.0        trans      9.009688  6.234898   
            2                2.0        trans      9.009688  6.234898   
            3                3.0        trans      9.009688  6.234898   
            4                4.0        trans      9.009688  6.234898   

field_name                                                             \
component                                                               
instance_name                                                           
idx                                               